**A ramsey experiment** is a fundemental quantum control technique used to measure the coherence time (T2*) of a qubit and precisely callibrate it's transition frequency. It isolates and tracks how a qubit's phase naturally evolves and dephases over time in the transverse plane of bloch sphere

To simulate a standard ramsey fringe experiment:
1. Prepare the qubit in |0>.
2. Apply a pi/2 pulse --> superposition (|0> + |1>)/sqrt(2)
3. Let it freely evolve (idle) for a variable delay time t, under a (possibly detuned) Hamiltonian and T1/T2 decoherence
4. Apply a second pi/2 "analysis" pulse.
5. Measure the population in |1>.
6. Repeat over a range of delays --> get an oscillating, decaying fringe pattern. Fit:


P1(t) = A*exp(-t/T2)*cos(2*pi *f *t+ phi) + C


to extract T2* (the free-induction dephasing time) and the detuning frequency f.

In [ ]:
%pip install qutip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 32.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import qutip as qt
from scipy.optimize import curve_fit

Validate T1

In [ ]:
def validate_T1(T1):
  if T1<=0:
    raise ValueError(f"T1 must be positive got: {T1}")

Validate T1 and T2

In [ ]:
def validate_T1_T2(T1, T2):
  validate_T1(T1)
  if T2<=0:
    raise ValueError(f"T2 must be positive got: {T2}")
  if T2>2*T1+1e-12:
    raise ValueError(f"T2 must be smaller than 2*T1 got: {T2}")

validate array of free-evolution delays on the experiment.

In [ ]:
def validate_delays(delays):
  delays = np.asarray(delays, dtype=float)
  if delays.size == 0:
    raise ValueError("delays cannot be empty")
  if delays.size < 6:
    raise ValueError(f"Need atleast 6 delay points to fit A, T2, f, phi, C; got {delays.size}")
  if np.any(delays < 0):
    raise ValueError("delays must be non-negative (idle time can't be negative).")
  if np.any(np.diff(delays) <= 0):
    raise ValueError("delays must be strictly increasing")
  return delays

validate the number of single-shot repetition per delay point

In [ ]:
def validate_shots(n_shots):
  if n_shots is not None:
    if n_shots <= 0:
      raise ValueError("n_shots must be positive")
    if not float(n_shots).is_integer():
      raise ValueError("n_shots must be an integer")

Warn (by raising) if the delay grid can't resolve the requested
    detuning, i.e. if the fringe period is shorter than ~2 sample
    spacings (Nyquist-style check). This is a common real-world
    Ramsey-experiment mistake: setting delays too coarse for the
    chosen detuning aliases the fringes.

In [ ]:
def validate_detuning_sampling(detuning, delays):
  if detuning==0:
    return
    # no oscillations
  dt = np.min(np.diff(delays))
  #calculates minimum time between delay points
  fringe_period = 1.0/abs(detuning)
  #calculates oscillation period
  if fringe_period < 2*dt:
    raise ValueError(
            f"Delay spacing (dt={dt:.4g}) is too coarse to resolve a fringe "
            f"period of {fringe_period:.4g} at detuning={detuning}. "
            "Use a finer delay grid or a smaller detuning."
        )
    #verifies Nyquist sampling theorem

building energy relaxation operator

Ramsey decay is limited to both T1 and pure dephasing. so this is included even though the sequence starts/ends via pulses rather than direct population prep.

In [ ]:
def relaxation_operator(T1):
  validate_T1(T1)
  gamma1 = 1.0/T1
  return np.sqrt(gamma1)*qt.sigmap()
  #L_relax = sqrt(1/T1)*sigma_minus

Building pure-dephasing collapse operator.

Ramsey T2 is directly sensitive to dephasing it's the whole point of measurement so this is required part of the ramsey collapse-operator set whenever T2<2*T1.

In [ ]:
def dephasing_operator(T1, T2):
  validate_T1_T2(T1, T2)
  gamma_phi = 1.0/T2 - 1.0/(2*T1)
  if gamma_phi ==0:
    return None
  return np.sqrt(gamma_phi/2)*qt.sigmaz()
  #L_dephase = sqrt(gamma_phi/2)*sigma_z

building hamiltonian

Unlike a T1 experiment, Ramsey is USUALLY run with a deliberate
    nonzero detuning (offset between drive and qubit frequency) so
    the fringes oscillate at a resolvable rate rather than sitting
    exactly at DC. detuning=0 is still valid (gives a non-oscillating
    decay envelope only).

In [ ]:
def build_hamiltonian(detuning=0.0):
  if detuning==0.0:
    return 0*qt.qeye(2)
    # 2X2 null matrix
  return 0.5*detuning*qt.sigmaz()

Assemble the collapse operators for the free-evolution period of a Ramsey experiment.

In [ ]:
def build_ramsey_collapse_operators(T1, T2):
  """Parameters
  T1: Energy relaxation time
  T2: Total dephasing time to be measured (T2<=2*T1)
  Returns
  c_ops"""
  validate_T1_T2(T1, T2)
  #verifies T1 and T2
  c_ops = [relaxation_operator(T1)]
  #creates T1 relaxation collapse operator list
  L_dephase = dephasing_operator(T1, T2)
  #builds pure dephasing collapse operator
  if L_dephase is not None:
    c_ops.append(L_dephase)
    #if L_dephase exists then adds to c_ops
  return c_ops

Build an ideal instantaneous single-qubit rotation pulse:

R_y(angle) = cos(angle/2)* I - i *sin(angle/2)*sigma_y

R_x(angle) = cos(angle/2)* I - i *sin(angle/2)*sigma_x

In [ ]:
def build_pulse_operator(angle, axis="y"):
  """Parameters
  angle: Rotation angle in radians
  axis: Rotation axis. Ramsey conventionally uses 'y' pulses to map |0> --> (|0>+|1>)/sqrt(2) with a real amplitude.
  Returns
  unitary pulse operator"""
  #a standard ramsey experiment uses a rotation angle of pi/2 but kept it as parameter for general use.
  #mostly y-axis is used because it returns real superposition state
  if axis  not in ("x", "y"):
    raise ValueError(f"axis must be 'x' or 'y' got: {axis}")
    #z axis is not included because z can only alter the phase of qubit and not it's population which doesn't lead to superposition of |0> state
  sigma = qt.sigmax() if axis == "x" else qt.sigmay()
  #checks the axis
  return np.cos(angle/2)*qt.qeye(2) - 1j*np.sin(angle/2)*sigma
  #returns single qubit rotation pulse- cos(angle/2)*I - i*sin(angle/2)*sigma

Apply an instantaneous unitary pulse to a density matrix: U* rho *U^dagger

In [ ]:
def apply_pulse(rho, U):
  return U*rho*U.dag()

simulate a ramsey T2 measurement:

|0> --[pi/2]--> free evolve (delay t) --[pi/2]--> measure P1

In [ ]:
def run_ramsey_experiment(T1, T2, delays, detuning=0.0, n_shots=None, seed=None):
  """Parameters
  T1: Energy relaxation time
  T2: True (simulated) total dephasing time to be measured, T2<=2*T1
  delays: Free-evolution idle times. Must be sorted, non-negative, atleast 6 points
  detuning: angular-frequency detuning during free evolution. Nonzero by convention in a real ramsey experiment,
  so the fringes are resolvable rahter than sitting at DC.
  n_shots: if given, simulate finite-sample projective measurement noise via binomial(n_shots, p1_ideal)/n_shots at each delay.
  seed: seed for noise RNG.
  Returns
  dict with keys:
  'delays' : np.ndarray of free-evolution times
  'pop_ideal': exact |1> population after the analysis pulse
  'pop_measured' : population used for fitting (noisy if n_shots given)
  'n_shots' : the n_shots value used (None if not applied)"""
  delays = validate_delays(delays)
  validate_shots(n_shots)
  validate_detuning_sampling(detuning, delays)
  #validate delays, n_shots, delay spacing
  c_ops = build_ramsey_collapse_operators(T1, T2)
  #builds collapse operators
  H = build_hamiltonian(detuning=detuning)
  U_pulse = build_pulse_operator(np.pi/2, axis = "y")
  #builds pi/2 rotation operator
  rho0 = qt.ket2dm(qt.basis(2, 0))
  #gives |0><0|
  rho_after_first_pulse = apply_pulse(rho0, U_pulse)
  #applies pi/2 pulse
  result = qt.mesolve(H, rho_after_first_pulse, delays, c_ops=c_ops)
  #master equation
  num_op = qt.num(2)
  #creates |1><1|
  pop_ideal = np.array([
      qt.expect(num_op, apply_pulse(state, U_pulse)) for state in result.states
  ])
  #Apply the analysis pulse and calculate the ideal excited-state population
  pop_ideal = np.clip(np.real(pop_ideal), 0.0, 1.0)
  #fixes range
  if n_shots is not None:
     rng = np.random.default_rng(seed)
     #creates a random generator
     counts = rng.binomial(n_shots, pop_ideal)
     #Generates noisy measurement counts using a binomial distribution.
     pop_measured = counts/n_shots
     #convert the sampled counts into measured excited-state probabilities
  else :
      pop_measured = pop_ideal.copy()
      #If shot noise is not simulated use ideal probabilities
  return {
        "delays": delays,
        "pop_ideal": pop_ideal,
        "pop_measured": pop_measured,
        "n_shots": n_shots,
    }



Damped-oscillation model: P1(t) = A*np.exp(-t/T2) * np.cos(2* np.pi *freq *t + phase) + C

In [ ]:
def _ramsey_model(t, A, T2, freq, phase, C):
  return A*np.exp(-t/T2)*np.cos(2*np.pi*freq*t + phase) + C

Estimate the fringe frequency via FFT(Fast Fourier Transform), for use as a curve_fit
    initial guess (a plain guess like 0 often fails to converge on
    real oscillatory data).

In [ ]:
def _estimate_frequency(delays, populations):
  detrended = populations - np.mean(populations)
  dt = np.mean(np.diff(delays))
  #calculates the difference of consequtive delays
  n = len(delays)
  fft_vals = np.abs(np.fft.rfft(detrended))
  #rfft = Real Fast Fourier Transform
  #converts Time-domain signal to frequency-domain signal
  fft_freqs = np.fft.rfftfreq(n, dt)
  #rfftfreq = Returns the list of frequencies corresponding
  if len(fft_freqs) <= 1:
    return 0.0
  #if there are no data points or less data points meaningful FFT is not possible so returns 0
  peak_idx = np.argmax(fft_vals[1:]) +1
  #skip DC bin
  return fft_freqs[peak_idx]

Fit measured ramsey fringe data to

P1(t) = A*exp(-t/T2) *cos(2* pi *f *t+ phi) + C

using nonlinear least squares

In [ ]:
def fit_T2_ramsey(delays, populations, p0=None):
  """Parameters
  delays : evolution times
  populations: measured excited-state probabilities
  p0 : Initial guess (A0, T2_0, f0, phi0, C0). Auto-derived(with an FFT-based guess) if not given.
  Returns
  dict with keys:
  'A', 'T2', 'freq', 'phase', 'C', 'A_err', 'T2_err', 'freq_err', 'phase_err', 'C_err'
  'pcov' """

  delays = np.asarray(delays, dtype=float)
  populations = np.asarray(populations, dtype=float)
  if delays.size != populations.size:
    raise ValueError("delays and populations must have the same size")

  if delays.size < 6:
    raise ValueError(f"Need atleast 6 delay points to fit A, T2, f, phi, C; got {delays.size}")
  if np.ptp(populations) < 1e-8 :
    raise ValueError(
        "Population data is flat (no observable fringes); cannot fit T2. "
            "Check that T2 is not much larger than the delay range, and that "
            "detuning is nonzero if you expect oscillation."
        )
  if p0 is None:
    C0 = np.mean(populations)
    #Assumes signal average as offset
    A0 = (np.max(populations) - np.min(populations)) / 2.0
    #estimates amplitude
    T2_0 = delays[-1] / 3.0 if delays[-1] > 0 else 1.0
    #divides last measured delay into 3 equal parts
    freq0 = _estimate_frequency(delays, populations)
    #estimates oscillation frequency
    phi0 = 0.0
    p0 = (A0 if A0 !=0 else 0.5, max(T2_0, 1e-6), freq0, phi0, C0)
    #final guess
  try:

          popt, pcov = curve_fit(
            _ramsey_model, delays, populations, p0=p0,
            bounds=([-2.0, 1e-9, -np.inf, -2 * np.pi, -1.0],
                    [2.0, np.inf, np.inf, 2 * np.pi, 1.0]),
            maxfev=20000,
     )
    #perform a bound nonlinear least-squares fit of the ramsey model.
  except RuntimeError as e:
        raise runtimeError(f"fit failed to converge: {e}") from e
  #Unpackthe optimized Ramsey model parameters returned by curve_fit.
  A, T2_fit, freq, phase, C = popt
  perr = np.sqrt(np.diag(pcov))
  return {
     "A": A, "T2": T2_fit, "freq": freq, "phase": phase, "C": C,
        "A_err": perr[0], "T2_err": perr[1], "freq_err": perr[2],
        "phase_err": perr[3], "C_err": perr[4],
        "pcov": pcov,
  }


In [ ]:
def save_ramsey_plot(delays, populations, fit_result, path="report/figures/ramsey_t2.png"):
    """Save a Ramsey fringe plot (data + fitted curve) to disk."""
    import os
    import matplotlib.pyplot as plt

    os.makedirs(os.path.dirname(path), exist_ok=True)

    t_fine = np.linspace(delays.min(), delays.max(), 600)
    fit_curve = _ramsey_model(
        t_fine, fit_result["A"], fit_result["T2"],
        fit_result["freq"], fit_result["phase"], fit_result["C"]
    )

    plt.figure(figsize=(6, 4))
    plt.plot(delays, populations, "o", label="measured", markersize=4)
    plt.plot(t_fine, fit_curve, "-",
              label=f"fit: T2* = {fit_result['T2']:.2f} \u00b1 {fit_result['T2_err']:.2f}, "
                    f"f = {fit_result['freq']:.4g}")
    plt.xlabel("Free evolution time")
    plt.ylabel("Population in |1>")
    plt.title("Ramsey T2* Measurement")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    return path

In [ ]:
if __name__ == "__main__":
    print("Ramsey T2 measurement experiment demo (standalone)")
    print("=" * 50)

    true_T1 = 80.0
    true_T2 = 30.0      # must be <= 2*T1 = 160
    true_detuning = 0.05  # rad/unit-time -> fringe period = 2*pi/0.05 ~ 126
    delays = np.linspace(0, 4 * true_T2, 80)

    L_relax = relaxation_operator(true_T1)
    L_dephase = dephasing_operator(true_T1, true_T2)
    H = build_hamiltonian(detuning=true_detuning)
    U_pulse = build_pulse_operator(np.pi / 2, axis="y")
    print(f"Relaxation operator L_relax (T1={true_T1}):\n{L_relax}\n")
    print(f"Dephasing operator L_dephase (T1={true_T1}, T2={true_T2}):\n{L_dephase}\n")
    print(f"Hamiltonian (detuning={true_detuning}):\n{H}\n")
    print(f"pi/2 pulse operator (y-axis):\n{U_pulse}\n")

    # Noiseless run
    data_ideal = run_ramsey_experiment(true_T1, true_T2, delays, detuning=true_detuning)
    fit_ideal = fit_T2_ramsey(data_ideal["delays"], data_ideal["pop_measured"])
    print(f"[Noiseless] True T2 = {true_T2}, Fitted T2 = {fit_ideal['T2']:.3f} "
          f"+/- {fit_ideal['T2_err']:.3f}")
    print(f"            True detuning = {true_detuning}, Fitted freq = {fit_ideal['freq']:.4f} "
          f"+/- {fit_ideal['freq_err']:.4f}")

    # Realistic run with finite-shot readout noise
    data_noisy = run_ramsey_experiment(true_T1, true_T2, delays, detuning=true_detuning,
                                        n_shots=2000, seed=42)
    fit_noisy = fit_T2_ramsey(data_noisy["delays"], data_noisy["pop_measured"])
    print(f"[2000 shots] Fitted T2 = {fit_noisy['T2']:.3f} +/- {fit_noisy['T2_err']:.3f}")

Ramsey T2 measurement experiment demo (standalone)
Relaxation operator L_relax (T1=80.0):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=False
Qobj data =
[[0.        0.1118034]
 [0.        0.       ]]

Dephasing operator L_dephase (T1=80.0, T2=30.0):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 0.11636867  0.        ]
 [ 0.         -0.11636867]]

Hamiltonian (detuning=0.05):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 0.025  0.   ]
 [ 0.    -0.025]]

pi/2 pulse operator (y-axis):
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=False
Qobj data =
[[ 0.70710678 -0.70710678]
 [ 0.70710678  0.70710678]]

[Noiseless] True T2 = 30.0, Fitted T2 = 30.000 +/- 0.000
            True detuning = 0.05, Fitted freq = 0.0080 +/- 0.0000
[2000 shots] Fitted T2 = 28.684 +/- 0.817
